# 🛡️ DeepGuard – EfficientNet-B0 Model Training Notebook
### Next-Gen JPEG-Invariant Deepfake Image & Video Forensic Detection

This notebook trains an **EfficientNet-B0** backbone on **Google Colab (with T4 GPU)** for binary deepfake classification (`0: Fake`, `1: Real`).

Key capabilities in this pipeline:
- **JPEG-Invariant Augmentation (`RandomJPEGAndScale`)**: Eliminates JPEG compression bias and artifact shortcuts from web downloads.
- **Balanced 50/50 Subsetting**: 6,000 Real + 6,000 Fake Train | 1,000 Real + 1,000 Fake Val for fast, unbiased training.
- **CosineAnnealingLR**: 6-epoch scheduled learning rate decay for optimal convergence.
- **Automatic Export**: Directly exports `deepguard_effnet.pth` ready for DeepGuard inference.

## 1. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Select Runtime -> Change runtime type -> T4 GPU!"
device = torch.device("cuda")
print("Connected to GPU:", torch.cuda.get_device_name(0))

## 2. Install Required Dependencies

In [ ]:
!pip install -q timm scikit-learn matplotlib torchvision kaggle

## 3. Dataset Setup (Kaggle "140k Real & Fake Faces")

If running in Colab for the first time, upload your `kaggle.json` API token when prompted.

In [ ]:
import io, os, glob, random, shutil
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
try:
    from google.colab import files  # type: ignore[import-not-found]  # pyrefly: ignore[missing-import]
except ImportError:
    files = None

# 1. Download dataset if not already extracted
if not os.path.exists("dataset/train/real"):
    if not os.path.exists("/root/.kaggle/kaggle.json") and files is not None:
        uploaded = files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
        os.chmod("/root/.kaggle/kaggle.json", 0o600)
    !pip install -q kaggle timm
    !kaggle datasets download -d xhlulu/140k-real-and-fake-faces
    !unzip -q 140k-real-and-fake-faces.zip -d raw_kaggle
    os.makedirs("dataset/train", exist_ok=True)
    os.makedirs("dataset/val", exist_ok=True)
    shutil.move(glob.glob("raw_kaggle/**/train/real", recursive=True)[0], "dataset/train/real")
    shutil.move(glob.glob("raw_kaggle/**/train/fake", recursive=True)[0], "dataset/train/fake")
    shutil.move(glob.glob("raw_kaggle/**/valid/real", recursive=True)[0], "dataset/val/real")
    shutil.move(glob.glob("raw_kaggle/**/valid/fake", recursive=True)[0], "dataset/val/fake")
    !rm -rf raw_kaggle 140k-real-and-fake-faces.zip
    print("Dataset extraction complete!")
else:
    print("Dataset already exists at ./dataset/")

## 4. Balanced 50/50 Fast Subset (6,000 Real + 6,000 Fake Train | 1,000 Real + 1,000 Fake Val)

In [ ]:
# 2. Balanced 50/50 subset (6,000 Real + 6,000 Fake Train | 1,000 Real + 1,000 Fake Val)
shutil.rmtree("fast_ds", ignore_errors=True)
for cls in ["real", "fake"]:
    for split, count in [("train", 6000), ("val", 1000)]:
        os.makedirs(f"fast_ds/{split}/{cls}", exist_ok=True)
        src_files = sorted(glob.glob(f"dataset/{split}/{cls}/*"))
        random.seed(42)
        for f in random.sample(src_files, min(count, len(src_files))):
            dst = f"fast_ds/{split}/{cls}/{os.path.basename(f)}"
            try:
                os.symlink(os.path.abspath(f), dst)
            except OSError:
                shutil.copy2(f, dst)

train_total = len(glob.glob("fast_ds/train/*/*"))
val_total = len(glob.glob("fast_ds/val/*/*"))
print(f"Fast dataset ready! Train samples: {train_total}, Val samples: {val_total}")

## 5. JPEG-Invariant Augmentation & Strict DataLoaders

In [ ]:
# 3. JPEG-Invariant Augmentation (eliminates JPEG compression bias on web downloads!)
class RandomJPEGAndScale:
    def __call__(self, img):
        if random.random() < 0.7:
            q = random.randint(45, 95)
            buf = io.BytesIO()
            img.save(buf, format="JPEG", quality=q)
            buf.seek(0)
            img = Image.open(buf).convert("RGB")
        if random.random() < 0.5:
            s = random.randint(128, 224)
            img = img.resize((s, s), Image.Resampling.BILINEAR)
        return img

train_tf = transforms.Compose([
    RandomJPEGAndScale(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_tf = transforms.Compose([
    RandomJPEGAndScale(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds = datasets.ImageFolder("fast_ds/train", transform=train_tf)
val_ds = datasets.ImageFolder("fast_ds/val", transform=val_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"DataLoaders initialized | Classes: {train_ds.class_to_idx}")

## 6. Train EfficientNet-B0 for 6 Epochs (Cosine Annealing Scheduler)

In [ ]:
# 4. Train EfficientNet-B0 for 6 Epochs
model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=6)

best_acc = 0.0
for epoch in range(1, 7):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    scheduler.step()

    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += y.size(0)
    acc = 100 * correct / total
    print(f"Epoch {epoch}/6 -> JPEG-Invariant Val Acc: {acc:.2f}%")
    if acc > best_acc:
        best_acc = acc
        torch.save(model.state_dict(), "deepguard_effnet.pth")

print(f"\nDone! Best Val Acc: {best_acc:.2f}%")

## 7. Export Model for DeepGuard Deployment

Download `deepguard_effnet.pth` and place it in your local DeepGuard directory:
`backend/weights/deepguard_effnet.pth`

In [ ]:
print("Downloading deepguard_effnet.pth...")
try:
    from google.colab import files  # type: ignore[import-not-found]  # pyrefly: ignore[missing-import]
    files.download("deepguard_effnet.pth")
    print("Triggered Colab file download!")
except ImportError:
    print("Local environment detected. Model saved at: ./deepguard_effnet.pth")